### Sequence-to-Sequence Models
Sequence-to-sequence (Seq2Seq) models are a type of neural network architecture designed to handle tasks where the input and output are both sequences, such as machine translation, text summarization,and speech recognition. These models typically consist of an encoder and a decoder.

The encoder processes the input sequence and encodes it into a fixed-length context vector, which captures the relevant information from the input. The decoder then takes this context vector and generates the output sequence one token at a time.

One of the key innovations in Seq2Seq models is the attention mechanism, which allows the decoder to focus on different parts of the input sequence when generating each token in the output. This has significantly improved the performance of Seq2Seq models, especially for longer sequences.

Overall, Seq2Seq models have become a fundamental architecture in natural language processing and have been widely used in various applications, including chatbots, language translation, and text generation.

Reference: https://www.geeksforgeeks.org/machine-learning/seq2seq-model-in-machine-learning/

Consider the formula: $h_t=σ(W^{hx}xt+W^{hh}h_{t−1})$
Where:
- $h_t$ is the hidden state at time step t
- $σ$ is the activation function (e.g., sigmoid or tanh)
- $W^{hx}$ is the weight matrix for the input at time step t
- $x_t$ is the input at time step t
- $W^{hh}$ is the weight matrix for the hidden state from the previous time step
- $h_{t−1}$ is the hidden state from the previous time step

And the formula: $y_t=σ(W^{hy}h_t)$
Where:
- $y_t$ is the output at time step t
- $σ$ is the activation function (e.g., sigmoid or softmax)
- $W^{hy}$ is the weight matrix for the hidden state to output transformation
- $h_t$ is the hidden state at time step t

We will apply them to a simple example of a Seq2Seq model for machine translation. Let's say we want to translate the English sentence (e.g., "I am learning") into French.

In [45]:
import numpy as np

In [46]:
def one_hot_encode(text):
    # Tokenize the sentence
    words = text.lower().split()
    # Build vocabulary
    vocab = sorted(set(words))
    vocab_index = {word: i for i, word in enumerate(vocab)}
    # Create one-hot matrix
    one_hot = np.zeros((len(words), len(vocab)))

    for i, word in enumerate(words):
        one_hot[i, vocab_index[word]] = 1

    return one_hot, vocab

In [47]:
def sequence_to_sequence(input_sequence, output_sequence, hidden_size=2):
    # Define the variables
    h_0 = np.zeros((1, hidden_size))  # Initial hidden state meaning 2 hidden units (columns)
    # x_t (size 1xn). W_hh = hidden_states (1x2) -> whh (n x 2)
    w_hx = np.random.rand(input_sequence.shape[1], hidden_size)  # Weights for input to hidden layer
    # h_{t-1} (1×2) · W_hh → (1×2)
    w_hh = np.random.rand(h_0.shape[1], hidden_size)  # Weights for hidden to hidden layer
    # y_t = h_t · W_yh → (1×2) · (2×1) → (1×1)
    w_yh = np.random.rand(h_0.shape[1], 1)  # Weights for hidden to output layer

    # Process the input sequence through the RNN
    for t in range(input_sequence.shape[0]):
        x_t = input_sequence[t].reshape(1, -1)  # Get the t-th input vector
        h_t = np.tanh(np.dot(x_t, w_hx) + np.dot(h_0, w_hh))  # Update hidden state using tanh activation
        y_t = np.dot(h_t, w_yh)  # Compute output
        print(f"Encoder time step {t}: Output shape: {y_t.shape}")
        h_0 = h_t  # Update hidden state for the next time step

    # Process the output sequence through the RNN (decoder)
    for t in range(output_sequence.shape[0]):
        x_t = output_sequence[t].reshape(1, -1)  # Get the t-th output vector
        h_t = np.tanh(np.dot(x_t, w_hx) + np.dot(h_0, w_hh))  # Update hidden state using tanh activation
        y_t = np.dot(h_t, w_yh)  # Compute output
        print(f"Decoder time step {t}: Output shape: {y_t.shape}")
        h_0 = h_t  # Update hidden state for the next time step

In [48]:
# Example to copy and paste into a Jupyter Notebook cell: "I am learning"
text = input("Enter a sentence: ")
input_sequence, vocab = one_hot_encode(text)
print("Vocabulary:", vocab)
print("Input sequence shape:", input_sequence.shape)
# Using the same sequence for both encoder and decoder for demonstration
sequence_to_sequence(input_sequence, input_sequence, hidden_size=2)  

Vocabulary: ['am', 'i', 'learning']
Input sequence shape: (3, 3)
Encoder time step 0: Output shape: (1, 1)
Encoder time step 1: Output shape: (1, 1)
Encoder time step 2: Output shape: (1, 1)
Decoder time step 0: Output shape: (1, 1)
Decoder time step 1: Output shape: (1, 1)
Decoder time step 2: Output shape: (1, 1)


Using PyTorch, we can define a simple Seq2Seq model as follows:

```python

In [49]:
import torch
import torch.nn as nn
import torch.nn.functional as F

#### Encoder

Each word in the input sentence is converted into a one-hot encoded vector, which is then passed through the encoder to produce a context vector.

In [50]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hidden_dim)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, hidden = self.rnn(embedded)
        return hidden

#### Decoder

The context vector from the encoder is used as the initial hidden state for the decoder, which generates the output sequence one token at a time.

In [51]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, input, hidden):
        input = input.unsqueeze(0)
        embedded = self.embedding(input)
        output, hidden = self.rnn(embedded, hidden)
        prediction = self.fc(output.squeeze(0))
        return prediction, hidden

#### Seq2Seq Model with Teacher Forcing
In training, we can use teacher forcing, where the actual target output is fed as the next input to the decoder instead of the predicted output. This helps the model learn faster.

In [52]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg=None, max_len=10, teacher_forcing_ratio=0.5):
        batch_size = src.shape[1]
        trg_vocab_size = self.decoder.fc.out_features
        outputs = []

        hidden = self.encoder(src)

        input = torch.zeros(batch_size, dtype=torch.long).to(self.device)

        for t in range(max_len):
            output, hidden = self.decoder(input, hidden)
            top1 = output.argmax(1)
            outputs.append(top1.unsqueeze(0))

            if trg is not None and t < trg.shape[0] and torch.rand(1).item() < teacher_forcing_ratio:
                input = trg[t]
            else:
                input = top1

        outputs = torch.cat(outputs, dim=0)
        return outputs

#### Usage Example with Outputs

In this example, we first one-hot encode the input sentence "I am learning" and then pass it through the Seq2Seq model. The model will generate the output sequence, which can be compared to the target French translation.

In [53]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VOCAB_SIZE = 10
EMB_DIM = 8
HID_DIM = 16
SEQ_LEN = 5
BATCH_SIZE = 2

enc = Encoder(VOCAB_SIZE, EMB_DIM, HID_DIM)
dec = Decoder(VOCAB_SIZE, EMB_DIM, HID_DIM)
model = Seq2Seq(enc, dec, device).to(device)

src = torch.randint(1, VOCAB_SIZE, (SEQ_LEN, BATCH_SIZE)).to(device)
trg = torch.randint(1, VOCAB_SIZE, (SEQ_LEN, BATCH_SIZE)).to(device)

outputs = model(src, trg, max_len=SEQ_LEN, teacher_forcing_ratio=0.7)

print("Source sequence (input tokens):")
print(src.T)
print("\nTarget sequence (true tokens):")
print(trg.T)
print("\nPredicted sequence (model output tokens):")
print(outputs.T)

Source sequence (input tokens):
tensor([[8, 7, 3, 5, 8],
        [7, 9, 3, 2, 5]])

Target sequence (true tokens):
tensor([[3, 7, 3, 3, 4],
        [5, 3, 9, 3, 4]])

Predicted sequence (model output tokens):
tensor([[7, 8, 7, 9, 8],
        [9, 8, 9, 8, 8]])


In [54]:
# Custom input and output sequences for demonstration
src = torch.tensor([[1, 2, 3, 4, 5],
                    [1, 2, 3, 4, 5]]).to(device)  # Example input sequence
trg = torch.tensor([[5, 4, 3, 2, 1],
                    [5, 4, 3, 2, 1]]).to(device)  # Example target sequence
outputs = model(src, trg, max_len=SEQ_LEN, teacher_forcing_ratio=0.7)
print("Custom Source sequence (input tokens):")
print(src.T)
print("\nCustom Target sequence (true tokens):")
print(trg.T)
print("\nCustom Predicted sequence (model output tokens):")
print(outputs.T)

Custom Source sequence (input tokens):
tensor([[1, 1],
        [2, 2],
        [3, 3],
        [4, 4],
        [5, 5]])

Custom Target sequence (true tokens):
tensor([[5, 5],
        [4, 4],
        [3, 3],
        [2, 2],
        [1, 1]])

Custom Predicted sequence (model output tokens):
tensor([[9, 9, 9, 8, 9],
        [9, 0, 0, 9, 8],
        [7, 0, 9, 8, 9],
        [9, 8, 8, 8, 9],
        [9, 9, 9, 8, 9]])
